In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize, integrate
from helper import *
import seaborn as sns
sns.set_style("darkgrid")

## Constants

In [ ]:
mu_sun= 1.327e11 ## km^3/s^2  
mu_earth= 3.986e5 ## km^3/s^2  # 3.986e5
mu_mars= 4.2830e4 ## km^3/s^2
mu_moon = 0.4902e4 ## km^3/s^2 Not given in paper

SOI_earth = 923502.24 # km
SOI_mars = 577723.87 # km 577723.87 
SOI_moon = 66300

D_mars = 2.279e8
D_earth = 1.496e8
D_moon = 384400 # km

r_LEO = 463
R_LEO = 6378.2 + r_LEO

w_earth = 1.99177621e-7 # rad/s
w_mars = 1.05850987e-7 # rad/s
w_moon = 2.6653e-6 # rad/s

r_LMO = 200
R_LMO = 3397.0 + r_LMO

## Geocentric Phase

In [4]:
def five_body_ODE_geo( t, state , init_phase, init_moon): # init phase in radians 
    rPEx, rPEy, vPEx, vPEy = state 
    rPE = np.array([rPEx, rPEy])
    rPE_norm = np.linalg.norm(rPE)

    rx_earth = D_earth*np.cos(w_earth*t)
    ry_earth = D_earth*np.sin(w_earth*t)
    r_earth = np.array([rx_earth, ry_earth])
    r_earth_norm = np.linalg.norm(r_earth)

    rx_mars = D_mars*np.cos(w_mars*t + init_phase)
    ry_mars = D_mars*np.sin(w_mars*t + init_phase)
    r_mars = np.array([rx_mars, ry_mars])

    vx_earth = -w_earth*D_earth*np.sin(w_earth*t)
    vy_earth = w_earth*D_earth*np.cos(w_earth*t)
    v_earth = np.array([vx_earth, vy_earth])

    vx_mars = -w_mars*D_mars*np.sin(w_mars*t + init_phase)
    vy_mars = w_mars*D_mars*np.cos(w_mars*t + init_phase)
    v_earth = np.array([vx_mars, vy_mars])

    rx_moon = D_moon*np.cos(w_moon*t + init_moon )
    ry_moon = D_moon*np.sin(w_moon*t + init_moon )
    r_moon = np.array([rx_moon, ry_moon])

    # Relative vectors (Spacecraft to Planet)
    rP = rPE + r_earth 
    rP_norm = np.linalg.norm(rP)

    rPM = rP - r_mars       # I think
    rPM_norm = np.linalg.norm(rPM)

    rPMoon = r_moon - rPE
    rPMoon_norm = np.linalg.norm(rPMoon)

    aPEx = (-((mu_sun / (rP_norm**3)) * (rPEx + rx_earth))
    - ((mu_earth / (rPE_norm**3)) * (rPEx))
    - ((mu_mars / (rPM_norm**3)) * (rPEx + rx_earth - rx_mars))
    - ((mu_moon / (rPMoon_norm**3)) * (rPEx - rx_moon))
    + ((mu_sun / (r_earth_norm**3)) * (rx_earth)))

    # aPEx = - ((mu_earth / (rPE_norm**3)) * (rPEx))
#     print(aPEx)
#     print(f"""
# {-((mu_sun / (rP_norm**3)) * (rPEx + rx_earth))}
# {- ((mu_earth / (rPE_norm**3)) * (rPEx))}
# {- ((mu_mars / (rPM_norm**3)) * (rPEx + rx_earth - rx_mars))}
# {+ (mu_sun / (r_earth_norm**3)) * (rx_earth)}
# """)

    aPEy = (-((mu_sun / (rP_norm**3)) * (rPEy + ry_earth)) 
    - ((mu_earth / (rPE_norm**3)) * (rPEy)) 
    - ((mu_mars / (rPM_norm**3)) * (rPEy + ry_earth - ry_mars)) 
    - ((mu_moon / (rPMoon_norm**3)) * (rPEy - ry_moon))
    + ((mu_sun / (r_earth_norm**3)) * (ry_earth)))

    # aPEy = - ((mu_earth / (rPE_norm**3)) * (rPEy))


#     print(f"""
# {-((mu_sun / (rP_norm**3)) * (rPEy + ry_earth))}
# {- ((mu_earth / (rPE_norm**3)) * (rPEy))}
# {- ((mu_mars / (rPM_norm**3)) * (rPEy + ry_earth - ry_mars))}
# {+ (mu_sun / (r_earth_norm**3)) * (ry_earth)}
# """)
    return [vPEx, vPEy, aPEx, aPEy]

# state = four_body_ODE_geo(0,y0 , 30)
# print(state)

## Heliocentric Phase

In [ ]:
def four_body_ODE_helio(t, state, init_phase): # init phase in radians 
    rPx, rPy, vPx, vPy = state 
    rP = np.array([rPx, rPy])
    rP_norm = np.linalg.norm(rP)

    rx_earth = D_earth*np.cos(w_earth*t)
    ry_earth = D_earth*np.sin(w_earth*t)
    r_earth = np.array([rx_earth, ry_earth])

    rx_mars = D_mars*np.cos(w_mars*t + init_phase)
    ry_mars = D_mars*np.sin(w_mars*t + init_phase)
    r_mars = np.array([rx_mars, ry_mars])

    vx_earth = -w_earth*D_earth*np.sin(w_earth*t)
    vy_earth = w_earth*D_earth*np.cos(w_earth*t)
    v_earth = np.array([vx_earth, vy_earth])

    vx_mars = -w_mars*D_mars*np.sin(w_mars*t + init_phase)
    vy_mars = w_mars*D_mars*np.cos(w_mars*t + init_phase)
    v_earth = np.array([vx_mars, vy_mars])

    rPM = rP - r_mars       # I think
    rPM_norm = np.linalg.norm(rPM)

    rPE = rP = r_earth
    rPE_norm = np.linalg.norm(rPE)

    aPx = (-((mu_sun / (rP_norm**3)) * (rPx))
    - ((mu_earth / (rPE_norm**3)) * (rPx - rx_earth))
    - ((mu_mars / (rPM_norm**3)) * (rPx - rx_mars)))

    aPy = (-((mu_sun / (rP_norm**3)) * (rPy))
    - ((mu_earth / (rPE_norm**3)) * (rPy - ry_earth))
    - ((mu_mars / (rPM_norm**3)) * (rPy - ry_mars)))

    return [vPx, vPy, aPx, aPy]

four_body_ODE_helio(0.5,[10,10,10,10], 10)

## Planetocentric Phase

In [ ]:
def four_body_ODE_mars( t, state, init_phase): # init phase in radians 
    rPMx, rPMy, vPMx, vPMy = state 
    rPM = np.array([rPMx, rPMy])
    rPM_norm = np.linalg.norm(rPM)

    rx_earth = D_earth*np.cos(w_earth*t)
    ry_earth = D_earth*np.sin(w_earth*t)
    r_earth = np.array([rx_earth, ry_earth])

    rx_mars = D_mars*np.cos(w_mars*t + init_phase)
    ry_mars = D_mars*np.sin(w_mars*t + init_phase)
    r_mars = np.array([rx_mars, ry_mars])
    r_mars_norm = np.linalg.norm(r_mars)

    vx_earth = -w_earth*D_earth*np.sin(w_earth*t)
    vy_earth = w_earth*D_earth*np.cos(w_earth*t)
    v_earth = np.array([vx_earth, vy_earth])

    vx_mars = -w_mars*D_mars*np.sin(w_mars*t + init_phase)
    vy_mars = w_mars*D_mars*np.cos(w_mars*t + init_phase)
    v_earth = np.array([vx_mars, vy_mars])

    rPE = rP = r_earth
    rPE_norm = np.linalg.norm(rPE)

    rP = rPE + r_earth 
    rP_norm = np.linalg.norm(rP)

    aPMx = -((mu_sun / (rP_norm**3)) * (rPMx + rx_mars))
    - ((mu_earth / (rPE_norm**3)) * (rPMx + rx_mars - rx_earth))
    - ((mu_mars / (rPM_norm**3)) * (rPMx))
    + ((mu_sun / (r_mars_norm**3)) * (rx_mars))

    aPMy = -((mu_sun / (rP_norm**3)) * (rPMy + ry_mars))
    - ((mu_earth / (rPE_norm**3)) * (rPMy + ry_mars - ry_earth))
    - ((mu_mars / (rPM_norm**3)) * (rPMy))
    + ((mu_sun / (r_mars_norm**3)) * (ry_mars))

    return [vPMx, vPMy, aPMx, aPMy]

four_body_ODE_mars(0.5,[10,10,10,10], 10)

## Integrate Functions

In [5]:
class Trajectory:
    def __init__(self, yPE_geo, t_geo, yP_helio, t_helio, yPM_mars, t_mars):
        self.yPE_geo = yPE_geo
        self.t_geo = t_geo
        self.yP_helio = yP_helio
        self.t_helio = t_helio
        self.yPM_mars = yPM_mars
        self.t_mars = t_mars

In [6]:
def integrate_geo(ys,ts, init_phase,dt = 10,step = 0):
    solver = integrate.ode(five_body_ODE_geo)
    solver.set_integrator("lsoda") #  
    solver.set_initial_value(ys, ts[0])
    solver.set_f_params(init_phase)
    # print(solver.t, ts)

    while solver.successful():
        solver.integrate(solver.t + dt)
        ys = np.vstack((ys,solver.y))
        ts = np.vstack((ts, solver.t))
        # print(ts)
        step += 1
        # print(ys[step,:2 ])
        # print(solver.get_return_code())
        if np.linalg.norm(ys[step,:2]) > SOI_earth:
            print(f"""
Exitted SOI of Earth at
rPE = {solver.y[0]} , {solver.y[1]} km
vPE = {solver.y[2] , solver.y[3]} km/s
t = {solver.t} s \n""")
            return ys, ts, True
        
        if step > 10000:
            print("""
Failed to exit Earth SOI""")
            return ys, ts, False
        

    print(ys)
    print(ts)


In [7]:
def integrate_helio(ys, ts,init_phase, dt = 300, step = 0):
    def get_rPM_norm(rP, t, debug = False): # Im gonna fucking crash out
        rx_mars = D_mars*np.cos(w_mars*t + init_phase)
        ry_mars = D_mars*np.sin(w_mars*t + init_phase)
        r_mars = np.array([rx_mars, ry_mars])
        rPM = rP - r_mars      # I think
        rPM_norm = np.linalg.norm(rPM)
        if debug:
            print(f"""
    {rx_mars = }
    {ry_mars = }
    {r_mars = }
    {rPM = }
    {rPM_norm = }""")
        return rPM_norm
    
    solver = integrate.ode(four_body_ODE_helio)
    solver.set_integrator("lsoda") #  
    solver.set_initial_value(ys, ts[0])
    solver.set_f_params(init_phase)
    print(ys[:2])
    distance_from_mars = [get_rPM_norm(ys[:2], ts[0])]
    while solver.successful() and step < (100000*8):
        solver.integrate(solver.t + dt)
        # print(solver.t)
        ys = np.vstack((ys,solver.y))
        ts = np.vstack((ts, solver.t))
        distance_from_mars = np.vstack((distance_from_mars, get_rPM_norm([solver.y[0],solver.y[1]], solver.t)))
        step += 1
        # print(f"{solver.y[0]:e} {solver.y[1]} {solver.t / (24*60*60)}")
        # print(f"{get_rPM_norm([solver.y[0],solver.y[1]], solver.t):e} {SOI_mars:e} {(get_rPM_norm([solver.y[0],solver.y[1]], solver.t) - SOI_mars):e} ")
        # if (get_rPM_norm([solver.y[0],solver.y[1]], solver.t) - SOI_mars) < 5e5:
        #     dt = 300

        if get_rPM_norm([solver.y[0],solver.y[1]], solver.t) < SOI_mars:
            print(f"""
Enterred SOI of Mars at
rP = {solver.y[0]:e} , {solver.y[1]:e} km
vP = {solver.y[2] , solver.y[3]} km/s
t = {solver.t} s\n""")
            return ys, ts, True, 0 
        
#         if solver.y[0] < 0:
#             if get_rPM_norm([ys[-1,0],ys[-1,1]], ts[-1]) < get_rPM_norm([ys[-2,0],ys[-2,1]], ts[-2]):
#                 print(f"""
# {get_rPM_norm([ys[-1,0],ys[-1,1]], ts[-1]) = }
# {get_rPM_norm([ys[-2,0],ys[-2,1]], ts[-2]) = }
# Swing and a miss """)
#                 return -1, ts

        if (solver.t/(60*60*24)) > 400:
            print(f"""
Failed to Enter Mars SOI
Closest to Mars at {np.min(distance_from_mars)}""")
            return ys, ts, False, np.min(distance_from_mars)

    print(ys, ts)

In [8]:
def integrate_mars(ys, ts,init_phase,deltaV_LMO, dt = 60, step = 0):
    solver = integrate.ode(four_body_ODE_mars)
    solver.set_integrator("lsoda") #  
    solver.set_initial_value(ys, ts[0])
    solver.set_f_params(init_phase)

    while solver.successful():
        solver.integrate(solver.t + dt)
        ys = np.vstack((ys,solver.y))
        ts = np.vstack((ts, solver.t))
        step += 1
        # print(ys[step,:2 ])
        # print(solver.get_return_code())
        if np.linalg.norm(ys[step,:2]) < R_LMO: # Not sure if this is a good way of doing it
            print(f"""
Reached desired height of Mars orbit at
rPM = {solver.y[0]} , {solver.y[1]} km
vPM = {solver.y[2] , solver.y[3]} km/s
t = {solver.t} s
{abs(g1(solver.y[0],solver.y[1],debug= True)) = }
{abs(g2(solver.y[2],solver.y[3], deltaV_LMO)) = }
{abs(g3(solver.y[0],solver.y[1],solver.y[2],solver.y[3], deltaV_LMO, orientation="counterclockwise")) = }\n""")
            return ys, ts, abs(g1(solver.y[0],solver.y[1])) + abs(g2(solver.y[2],solver.y[3], deltaV_LMO)) + abs(g3(solver.y[0],solver.y[1],solver.y[2],solver.y[3], deltaV_LMO, orientation="counterclockwise"))
            # return ys, ts, g(solver.y[0],solver.y[1],solver.y[2],solver.y[3], deltaV_LMO, debug= True)
        
        if np.linalg.norm(ys[step,:2]) > np.linalg.norm(ys[step - 1,:2]):
            print(f"""
Swing and a miss
Closest distance = {np.linalg.norm(ys[step - 1,:2])} km""")
            return ys, ts, abs(g1(solver.y[0],solver.y[1])) + abs(g2(solver.y[2],solver.y[3], deltaV_LMO)) + abs(g3(solver.y[0],solver.y[1],solver.y[2],solver.y[3], deltaV_LMO, orientation="counterclockwise"))
            # return ys, ts, g(solver.y[0],solver.y[1],solver.y[2],solver.y[3], deltaV_LMO)
        if step > 8000*3:
            print(f"Failed to solve for constraints in time")
            return -1, ts, np.inf

    # if g(solver.y[0], solver.y[1], solver.y[2], solver.y[3], deltaV_LMO):
    #     break

# print(ys)
# print(ts)

## Simulate Function

In [9]:
def simulate(thetaPE0, init_phase, deltaV_LEO, deltaV_LMO): # Give angles in degrees
    # Success Flags
    geo_success = False
    helio_success = False
    mars_success = False

    # Initialize trajectory
    yPE_geo = t_geo = yP_helio = t_helio = yPM_mars = t_mars = 0

    ############GEOCENTRIC#####################
    # timestep 
    dt = 60.0

    # Parameters
    thetaPE0 = np.deg2rad(thetaPE0)
    init_phase = np.deg2rad(init_phase)
    deltaV_LEO = deltaV_LEO
    deltaV_LMO = deltaV_LMO

    # IC for geocentric phase
    rPEx0 = R_LEO*np.cos(thetaPE0) # thetaPE0 to be perscribed
    rPEy0 = R_LEO*np.sin(thetaPE0) # thetaPE0 to be perscribed

    vPEx0 = -(np.sqrt(mu_earth/R_LEO) + deltaV_LEO) * np.sin(thetaPE0)
    vPEy0 = (np.sqrt(mu_earth/R_LEO) + deltaV_LEO) * np.cos(thetaPE0)
    # print(np.sqrt(mu_earth/R_LEO) * np.sin(thetaPE0))
    # print(vPEy0)

    y0 =np.array([rPEx0, rPEy0, vPEx0, vPEy0], None)
    ys = y0
    ts = [0]
    print(ys, ts)
    yPE_geo, t_geo, geo_success = integrate_geo(ys, ts, init_phase, dt=dt)
    if geo_success == False:
        objective = 20e50

    #################HELIOCENTRIC#####################
    
    if geo_success:
        # timestep 
        dt = 300

        t1 = t_geo[-1]
        rPEx1, rPEy1, vPEx1, vPEy1 = yPE_geo[-1]
        # Earth Pos and Vel
        rx_earth1 = D_earth*np.cos(w_earth*t1)
        ry_earth1 = D_earth*np.sin(w_earth*t1)

        vx_earth1 = -w_earth*D_earth*np.sin(w_earth*t1)
        vy_earth1 = w_earth*D_earth*np.cos(w_earth*t1)
    
        # initial conditions
        rPx1 = rx_earth1 + rPEx1 
        rPy1 = ry_earth1 + rPEy1 

        vPx1 = vx_earth1 + vPEx1 
        vPy1 = vy_earth1 + vPEy1 

        y1 =np.array([rPx1[0], rPy1[0], vPx1[0], vPy1[0]])

        ys = y1
        ts = t1
        yP_helio, t_helio, helio_success, closest_to_mars = integrate_helio(ys, ts, init_phase, dt = dt)
        if helio_success == False:
            objective = closest_to_mars**2
        # print(f"{yP_helio} test")
        # if np.size(yP_helio) == 1:
        #     objective = dist_from_mars**2
        #     helio_success = False
        # else:
        #     helio_success = True

    ###################MARSCENTRIC########################
    if helio_success:
        # timestep
        dt = 60

        rPx2, rPy2, vPx2, vPy2 = yP_helio[-1]
        t2 = t_helio[-1]

        # Mars Pos and Vel
        rx_mars2 = D_mars*np.cos(w_mars*t2 + init_phase)
        ry_mars2 = D_mars*np.sin(w_mars*t2 + init_phase)

        vx_mars2 = -w_mars*D_mars*np.sin(w_mars*t2 + init_phase)
        vy_mars2 = w_mars*D_mars*np.cos(w_mars*t2 + init_phase)

        # Initial Conditions
        rPMx2 = rPx2 - rx_mars2
        rPMy2 = rPy2 - ry_mars2

        vPMx2 = vPx2 - vx_mars2
        vPMy2 = vPy2 - vy_mars2

        y2 =np.array([rPMx2[0], rPMy2[0], vPMx2[0], vPMy2[0]], None)

        ys = y2
        ts = t2

        # print(ys , ts)
        yPM_mars, t_mars, objective = integrate_mars(ys, ts,init_phase, deltaV_LMO ,dt = dt)
        if np.size(yPM_mars) == 1:
            objective = np.inf
            mars_success = False
        else:
            mars_success = True
            print(f"""
{deltaV_LEO = }
{deltaV_LMO = }""")
    return objective, Trajectory(yPE_geo, t_geo, yP_helio, t_helio, yPM_mars, t_mars)


